# Phase 4 (Diarization) on Colab T4

Runs `src/diarization/diarize.py` (pyannote.audio speaker diarization + target-speaker isolation) against a T4 GPU.

The code is already verified end-to-end on one real file (2026-08-06, on a local T1000 box) -- this notebook runs the same code against the full 199-file corpus on Colab instead.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. **Accept the gated model terms** (one-time, needs your Hugging Face account, can't be automated):
   - Visit https://huggingface.co/pyannote/speaker-diarization-3.1 and accept the terms.
   - Visit https://huggingface.co/pyannote/segmentation-3.0 and accept the terms (a dependency of the pipeline above).
   - Create a read-scoped access token at https://huggingface.co/settings/tokens if you don't have one.
3. Add that token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar -> "Add new secret") so it's never pasted in plaintext. If you skip this, the notebook will fall back to a hidden `getpass` prompt instead.
4. Upload `data/raw/audio/` (199 files) **and** `data/transcripts/` (199 files -- diarization needs the word-level timestamps from Phase 3 to align speakers to words) to a folder in your Google Drive.
   - If you already uploaded `data/raw/audio/` for the Phase 3 notebook, you can reuse the same Drive folder.
   - Double check `data/transcripts/` in Drive actually has all 199 files, not just the 155 that finished before the Phase 3 Colab session got cut off by the daily quota -- the complete 199 only exists locally (or on the T1000) unless you've re-synced it since.
5. Edit `DRIVE_AUDIO_DIR`, `DRIVE_TRANSCRIPTS_DIR`, and `DRIVE_DIARIZED_DIR` in the **Configure paths** cell below to match your Drive layout.

Output lands directly in the Drive-mounted diarized folder, so there's no separate download step.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv

## 2. Clone the repo

In [ ]:
!git clone https://github.com/DAG-21/PureBillion-Cloner.git
%cd PureBillion-Cloner
!git log --oneline -5

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure paths

Edit these to match where you uploaded/want the data in your Drive, then run the cell.

In [ ]:
# EDIT THESE to match your Drive layout
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/persona-clone/data/raw/audio"
DRIVE_TRANSCRIPTS_DIR = "/content/drive/MyDrive/persona-clone/data/transcripts"
DRIVE_DIARIZED_DIR = "/content/drive/MyDrive/persona-clone/data/diarized"
DRIVE_METADATA_DIR = "/content/drive/MyDrive/persona-clone/data/raw/metadata"  # used for the corpus-time estimate only

import os
assert os.path.isdir(DRIVE_AUDIO_DIR), f"Not found: {DRIVE_AUDIO_DIR} -- upload your audio there first"
assert os.path.isdir(DRIVE_TRANSCRIPTS_DIR), f"Not found: {DRIVE_TRANSCRIPTS_DIR} -- upload data/transcripts/ there first"
os.makedirs(DRIVE_DIARIZED_DIR, exist_ok=True)

n_audio = len(os.listdir(DRIVE_AUDIO_DIR))
n_transcripts = len(os.listdir(DRIVE_TRANSCRIPTS_DIR))
print("Audio files found:", n_audio)
print("Transcript files found:", n_transcripts)
if n_transcripts < n_audio:
    print(f"WARNING: fewer transcripts ({n_transcripts}) than audio files ({n_audio}) -- "
          "some files will be skipped as missing_transcript. Make sure the full 199-file "
          "data/transcripts/ was uploaded, not a partial Colab-only copy.")

## 5. Install dependencies

Colab already ships a CUDA-compatible `torch`, so this shouldn't force a torch reinstall -- but if imports fail right after installing, use `Runtime -> Restart session` and re-run from here (a known Colab gotcha: newly installed packages sometimes aren't picked up until the runtime restarts).

In [ ]:
!pip install -q pyannote.audio python-dotenv pyyaml tqdm

## 6. Set HF_TOKEN

Reads the `HF_TOKEN` Colab secret if you added one (step 3 above); otherwise prompts for it via a hidden `getpass` input. Either way, the token is never printed or committed anywhere.

In [ ]:
import os

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN not found in Colab secrets -- paste it here (input hidden): ")

assert HF_TOKEN, "HF_TOKEN is required -- see the prerequisites at the top of this notebook"
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"HF_TOKEN set ({len(HF_TOKEN)} chars)")

## 7. Sanity-check gated model access

Loads the pipeline once, up front, so a missing token or un-accepted license terms fail here with a clear message -- instead of partway through the full batch run. Also caches the model weights for every later cell.

In [ ]:
from pyannote.audio import Pipeline

PIPELINE_NAME = "pyannote/speaker-diarization-3.1"

try:
    _pipeline = Pipeline.from_pretrained(PIPELINE_NAME, token=os.environ["HF_TOKEN"])
    print("Pipeline loaded OK:", PIPELINE_NAME)
    del _pipeline  # just checking access here; diarize.py loads its own copy
except Exception as exc:
    raise RuntimeError(
        "Failed to load the gated pyannote pipeline. Make sure your Hugging Face account "
        "has accepted the license terms at https://huggingface.co/pyannote/speaker-diarization-3.1 "
        "and https://huggingface.co/pyannote/segmentation-3.0, and that HF_TOKEN is a valid "
        "access token for that same account."
    ) from exc

## 8. Dry run

Confirms the file list (and that every audio file has a matching transcript) without loading the GPU pipeline -- same check already done locally on the Dell laptop.

In [ ]:
!python -m src.diarization.diarize \
  --audio-dir "{DRIVE_AUDIO_DIR}" \
  --transcripts-dir "{DRIVE_TRANSCRIPTS_DIR}" \
  --output-dir "{DRIVE_DIARIZED_DIR}" \
  --dry-run

## 9. Single-file benchmark

Diarizes one real file (with its matching transcript) and times it, mirroring the Phase 3 RTF benchmark -- gives a real number to sanity-check before committing GPU time to the full 199-file batch. Runs against scratch dirs so it doesn't touch the real `diarization_history.csv` / skip-if-exists state for the full run below.

In [ ]:
import glob, os, shutil, tempfile, time

audio_files = sorted(glob.glob(os.path.join(DRIVE_AUDIO_DIR, "*")))
assert audio_files, "No audio files found in DRIVE_AUDIO_DIR"

# pick the first audio file that actually has a matching transcript
sample_file = None
for candidate in audio_files:
    video_id = os.path.splitext(os.path.basename(candidate))[0]
    if os.path.exists(os.path.join(DRIVE_TRANSCRIPTS_DIR, f"{video_id}.json")):
        sample_file = candidate
        break
assert sample_file, "None of the audio files have a matching transcript -- check DRIVE_TRANSCRIPTS_DIR"
video_id = os.path.splitext(os.path.basename(sample_file))[0]
print("Benchmarking on:", sample_file)

bench_audio_dir = tempfile.mkdtemp()
bench_transcripts_dir = tempfile.mkdtemp()
bench_output_dir = tempfile.mkdtemp()
shutil.copy(sample_file, bench_audio_dir)
shutil.copy(os.path.join(DRIVE_TRANSCRIPTS_DIR, f"{video_id}.json"), bench_transcripts_dir)

In [ ]:
start = time.time()
!python -m src.diarization.diarize \
  --audio-dir "{bench_audio_dir}" \
  --transcripts-dir "{bench_transcripts_dir}" \
  --output-dir "{bench_output_dir}" \
  --history-file "{bench_output_dir}/bench_history.csv" \
  --device cuda
elapsed = time.time() - start
print(f"Wall clock: {elapsed:.1f}s")

In [ ]:
import json

diarized_path = glob.glob(os.path.join(bench_output_dir, "*.json"))[0]
with open(diarized_path) as f:
    result = json.load(f)

audio_duration = result["duration"]
rtf = elapsed / audio_duration
print(f"Audio duration: {audio_duration:.1f}s")
print(f"Diarization wall clock: {elapsed:.1f}s")
print(f"RTF (wall_clock / audio_duration): {rtf:.3f}")
print(f"-> roughly {1/rtf:.1f}x real-time on this T4")
print(f"Target speaker identified as: {result['target_speaker']}")
print(f"Speakers found: {result['speakers']}")

In [ ]:
# Extrapolate to the full corpus, if metadata is available
if os.path.isdir(DRIVE_METADATA_DIR):
    total_duration = 0.0
    for meta_file in glob.glob(os.path.join(DRIVE_METADATA_DIR, "*.json")):
        with open(meta_file) as f:
            meta = json.load(f)
        total_duration += meta.get("duration", 0)
    est_seconds = total_duration * rtf
    print(f"Total corpus duration: {total_duration/3600:.1f} hours")
    print(f"Estimated full-batch diarization time on this T4: {est_seconds/3600:.1f} hours")
else:
    print(f"DRIVE_METADATA_DIR not found ({DRIVE_METADATA_DIR}) -- skipping full-corpus estimate.")

## 10. Full batch run

Only run this once the estimate above looks reasonable for a single Colab session (free tier disconnects on ~90 min idle and caps sessions around ~12h -- if the estimate exceeds that, either upgrade to Colab Pro or split the batch across multiple sessions, since the pipeline already skips already-diarized IDs on rerun).

Passes `--history-file` explicitly into the Drive-mounted output dir -- the Phase 3 notebook's full-run cell skipped this and lost its history CSV to the ephemeral Colab VM disk on disconnect. Harmless there since file-existence is also checked, but worth doing right here since it's a one-line fix.

In [ ]:
!python -m src.diarization.diarize \
  --audio-dir "{DRIVE_AUDIO_DIR}" \
  --transcripts-dir "{DRIVE_TRANSCRIPTS_DIR}" \
  --output-dir "{DRIVE_DIARIZED_DIR}" \
  --history-file "{DRIVE_DIARIZED_DIR}/diarization_history.csv" \
  --device cuda

## 11. Sanity check the output

In [ ]:
import glob

diarized_files = glob.glob(os.path.join(DRIVE_DIARIZED_DIR, "*.json"))
print(f"{len(diarized_files)} diarized JSON files in {DRIVE_DIARIZED_DIR}")

with open(diarized_files[0]) as f:
    sample = json.load(f)
print("\nSample file:", diarized_files[0])
print("Speakers:", sample["speakers"])
print("Target speaker:", sample["target_speaker"])
print("Target speaker text (first 300 chars):", sample["target_speaker_text"][:300])

## Next steps

- Sync `DRIVE_DIARIZED_DIR` back down to local (or to the T1000 machine) -- `data/diarized/` is gitignored, so this only travels manually, same as `data/raw/` and `data/transcripts/` before it.
- Spot-check a handful of `target_speaker_text` fields against the source audio to confirm the longest-total-duration heuristic picked the right speaker in files with more than 2 speakers, or unusual talk-time balance -- it was only verified on 1 file so far.
- Update `PROJECT_UPDATES.md` with the real RTF/timing numbers from step 9 and the full-batch result once you have them.
- Once diarization output looks good, Phase 5 (cleaning) is next: normalizing `target_speaker_text` / `target_speaker_segments` into the cleaned transcripts the rest of the pipeline (chunking, embeddings, dataset_gen) will build on.